# Denoising 1: PCA Residuals → GDBN

**Goal:** clean the returns before the GDBN looks for connections.

On any day, most of a stock's move is **shared** with many other stocks (the market, its sector, "volatile stocks fell today"). Those shared moves make everything look connected. If they are removed, what is left is each stock's **own surprise**, and an arrow in the GDBN then means a **direct** link.

```
today's excess returns (~600 stocks)
        |
        |  PCA: the K directions that explain most of the day-to-day moves (fit on the past 5 years)
        v
shared part = loadings x K factor returns     +     stock-specific part (the RESIDUAL)
                      |                                          |
          K factor returns -> GDBN market nodes          residuals -> GDBN stock nodes
```

**PCA is the simplest denoiser** (a linear autoencoder with a waist of K numbers). The other notebooks must beat it.

| | Setting |
|---|---|
| Input | `exret_1d` from the feature table (each stock's excess return vs the S&P 500 average), all stocks with data that day |
| Fit | for each year, PCA on the previous 5 years (stocks with returns on at least 60% of those days) |
| Waist | **K = 5** factors |
| Each day | factor returns = cross-sectional regression of that day's returns on the loadings; residual = return − loadings × factors |
| Limitation | a stock with too little history in the window has no loadings, so its "residual" is just its raw return (IPCA, notebook 15, fixes this) |

Takes about 10–15 minutes the first time (the two raw GDBNs are saved for later notebooks).

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pit_common as pc
import denoise_common as dc

D = pc.load()                        # returns and universe for the GDBN (everything after 2022 blanked)
cube = dc.load_cube()                # the feature table (features_daily.csv.gz) as arrays
GDBN_MASK = dc.gdbn_mask(D, cube)    # the GDBN's 52-stock universe, for R^2 on its own stocks


def frame(arr, rows, cols=None):
    return pd.DataFrame(arr, index=cube.days[rows], columns=cube.perm if cols is None else cols)

## 1. Fit PCA each year and split returns into factor part + residual

In [ ]:
K, MIN_COVER = 5, 0.6
by_year, rows = {}, []
t0 = time.time()
for Y in dc.YEARS:
    fd, ad = dc.fit_days(cube, Y), dc.apply_days(cube, Y)
    A = np.where(cube.AVAIL[fd], cube.R[fd], np.nan)
    keep = np.where(np.isfinite(A).mean(axis=0) >= MIN_COVER)[0]          # stocks with enough history
    A = np.nan_to_num(A[:, keep])
    A -= A.mean(axis=0)
    _, s, Vt = np.linalg.svd(A, full_matrices=False)
    B = np.zeros((len(cube.perm), K))
    B[keep] = Vt[:K].T                                                     # loadings (stock x factor)

    resid = np.full((len(ad), len(cube.perm)), np.nan, dtype=np.float32)
    F = np.zeros((len(ad), K))
    has_load = np.zeros(len(cube.perm), dtype=bool); has_load[keep] = True
    for n, d in enumerate(ad):
        m = cube.AVAIL[d]
        h = m & has_load
        f = np.linalg.lstsq(B[h], cube.R[d, h], rcond=None)[0]               # today's factor returns
        resid[n, m] = cube.R[d, m]
        resid[n, h] -= B[h] @ f
        F[n] = f
    by_year[Y] = SimpleNamespace(RET=frame(resid, ad), SENT=None, ATTN=None,
                                 FACT=pd.DataFrame(F, index=cube.days[ad], columns=[f"pca{k + 1}" for k in range(K)]))

    yd = cube.days[ad].year == Y                                            # out-of-sample days for this year's PCA
    actual, fitted = cube.R[ad][yd], (cube.R[ad] - resid)[yd]
    rows.append({"year": Y, "stocks with loadings": len(keep),
                 "fit: variance explained": (s[:K] ** 2).sum() / (s ** 2).sum(),
                 "R2 S&P members (out of sample)": dc.r2(np.where(cube.MEM[ad][yd], actual, np.nan), fitted),
                 "R2 GDBN universe (out of sample)": dc.r2(np.where(GDBN_MASK[ad][yd], actual, np.nan), fitted),
                 "members without loadings": 1 - (cube.MEM[ad][yd] & has_load).sum() / cube.MEM[ad][yd].sum()})
r2_table = pd.DataFrame(rows).set_index("year")
print(f"Done in {time.time() - t0:.0f}s")
r2_table.round(3)

**Reading it:** **R²** = share of each day's excess-return moves explained by the 5 factors, on days the PCA was **not** fit on. This is the part removed as "shared"; the residual (1 − R²) is what the GDBN now sees. Higher R² in crisis years (2008–2009, 2020) is expected: stocks move together more when markets are stressed.

## GDBN on the denoised data

Four versions of the GDBN, all on the same 52-stock point-in-time universe, walk-forward, λ rule (best rank IC on 2006–2020), and 2021–2022 validation:

| Version | Inputs (yesterday's values) |
|---|---|
| **raw daily** | excess return, sentiment, attention, ETFs (notebook 06) |
| **raw slow** | past-week and past-month averages of the same (notebook 10, the best so far) |
| **PCA daily** | the same, with the **denoised** series in place of the raw one (the residual return), plus the 5 PCA factor returns as market nodes |
| **PCA slow** | past-week and past-month averages of the denoised series, plus the factors' averages |

The target is unchanged: each stock's next-day excess return. The raw versions are saved the first time and loaded in later notebooks.

**Columns:** `arrows/quarter` = network size at the chosen λ; `arrow stability` = how much consecutive quarters' networks overlap (1 = identical). Denser networks overlap more just by chance, so `arrows @ raw λ` and `stability @ raw λ` repeat both at the **same λ as the raw GDBN** of the same kind: the fair, like-for-like comparison. Then rank IC, Newey–West t, gross Sharpe (before costs), and breakeven bps (the cost level that would wipe out the profit) for each period.

In [ ]:
runs = dc.raw_gdbn(D)
runs["PCA daily"] = dc.walk_forward("pca_daily", D, dc.gdbn_features(by_year, "daily"))
runs["PCA slow"] = dc.walk_forward("pca_slow", D, dc.gdbn_features(by_year, "slow"))
table, scores = dc.evaluate(D, runs)
print(table.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, s in scores.items():
    ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
    axes[0].plot(ic.cumsum(), label=name, ls="--" if name.startswith("raw") else "-")
for name, res in runs.items():
    k, _ = pc.choose_lambda(D, res, 1)
    axes[1].plot([q["r"] for q in res], [(q["W"][:, :, k] != 0).sum() for q in res], label=name,
                 ls="--" if name.startswith("raw") else "-")
axes[0].axvline(pd.Timestamp("2021-01-01"), color="gray", lw=1); axes[0].set_title("Cumulative rank IC (rising = skill; line = validation starts)")
axes[1].set_title("Arrows in each quarter's network (at each version's chosen λ)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading it:**
- **Did denoising help prediction?** Compare the PCA rows with the raw rows of the same kind (daily vs daily, slow vs slow): rank IC, gross Sharpe, breakeven, and especially **2021–2022**.
- **Did it clean up the connections?** Compare **stability @ raw λ** (same λ, so the same price per arrow): higher means the network keeps finding the same links from quarter to quarter, which is what real (not noise-driven) connections should do. The plain `arrow stability` column is inflated for versions that chose a smaller λ (bigger networks).
- Notebook 18 puts all four denoisers side by side.

In [ ]:
dc.save("pca", table, r2_table)